# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.01 · Etiquetado Ollama local o Hugging Face en Colab

Usa Ollama HTTP en local o, opcionalmente, Hugging Face sobre Colab L4; ambos conservan el mismo contrato y reanudan por `chunk_id`.

Ollama admite un JSON Schema en la salida estructurada y su validación posterior con Pydantic [1]. El backend local usa exactamente `qwen3.5:4b` y debe registrar el digest publicado por Ollama [2]; el backend Colab usa `Qwen/Qwen3-4B`, cuyo linaje se describe en el informe Qwen3 [3] y cuya revisión exacta consta en la tarjeta del modelo [4]. Las salidas LLM son propuestas de anotación y no *ground truth*: la anotación asistida en tareas subjetivas requiere controles humanos y de anclaje [5]. El prompt, el reintento y la precedencia son decisiones locales.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab L4 desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA L4**. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. Si la copia activa no coincide, la celda lee `bundle_releases/latest.json`, verifica todos sus SHA-256 y promueve automáticamente esa versión. Ejecute antes `02_00` directamente en Colab. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [6]. La integridad del bundle se comprueba con SHA-256 [7]. No sincronice cachés de modelos ni escriba checkpoints directamente en Drive.

In [5]:
# Backend reproducible: local o Google Colab desde VS Code
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_NOTEBOOK_ID = "02_01"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "7d6707a82846e70eafb1e443f698294b70dc4d157a16295224962e394517bf29"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "0f93de9d42cf6ec6a0e7a5d90e82d12a408c5c6c094f03f85d61718f85b26843"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if not manifest_path.is_file():
        return False
    try:
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            return False
        if manifest["bundle_id"] != expected_bundle_id:
            return False
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            return False
        return all(
            (bundle_dir / name).is_file() and _sha256(bundle_dir / name) == expected_sha256
            for name, expected_sha256 in _bundle_specs(manifest)
        )
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    if not latest_pointer_path.is_file():
        raise FileNotFoundError(
            "Falta bundle_releases/latest.json. Ejecute 02_00_preparacion_bundle_colab.ipynb "
            "directamente en Colab y confirme la publicación en Drive."
        )
    latest_pointer = _read_manifest(latest_pointer_path)
    latest_bundle_id = str(latest_pointer.get("bundle_id") or "")
    if len(latest_bundle_id) != 64:
        raise ValueError("bundle_releases/latest.json no contiene un bundle_id válido")
    if latest_pointer.get("core_sha256") != COLAB_EXPECTED_CORE_SHA256:
        raise RuntimeError(
            "Drive contiene una versión de código distinta de la esperada por este notebook. "
            "Regénere los cuadernos desde la misma versión local que publicó 02_00 y vuelva a abrirlos."
        )
    RELEASE_DIR = RELEASES_DIR / latest_bundle_id
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    if not release_manifest_path.is_file() or _sha256(release_manifest_path) != latest_pointer.get("manifest_sha256"):
        raise RuntimeError("El manifiesto de la versión latest de Drive falta o no coincide con su puntero")
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive. Ejecute en Colab "
                "02_00_preparacion_bundle_colab.ipynb, confirme status=published_to_drive y "
                f"compruebe que exista {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


estado,ya_estaba_actualizado
bundle_id,d6f855632fc644960d7042e9605f0d239e02bc928c9ce301d6583f49cb50cc34
bundle_del_notebook_al_generarse,d6f855632fc644960d7042e9605f0d239e02bc928c9ce301d6583f49cb50cc34
core_sha256,697279cee717eb099d44f2e8d7d77dc36fec5817e322d2757c3d6341c150b638
generado,2026-08-07T19:03:16.162384+00:00
versión_inmutable_drive,Ver detalle/content/drive/MyDrive/ModeracionPeru_Colab/bundle_releases/d6f855632fc644960d7042e9605f0d239e02bc928c9ce301d6583f49cb50cc34


is_colab,Sí
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""auto"", ""device_name"": ""NVIDIA L4"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 23659151360, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
nvidia_smi,"NVIDIA L4, 23034 MiB, 580.82.07"
cwd,/content
free_runtime_bytes,70137102336


notebook_id,02_01
run_id,02_01_working_v2_1
drive_root,/content/drive/MyDrive/ModeracionPeru_Colab
runtime_root,/content/moderacion_peru
project_root,/content/moderacion_peru/project
input_paths,"Ver detalle{ ""chunks_v2"": ""/content/moderacion_peru/inputs/datos/processed/chunks_v2.jsonl"" }"
scratch_output_dir,/content/moderacion_peru/runs/02_01/02_01_working_v2_1
drive_run_dir,/content/drive/MyDrive/ModeracionPeru_Colab/runs/02_01/02_01_working_v2_1
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""cuda"", ""device_name"": ""NVIDIA L4"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 23659151360, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
resumed,No


## Selección del proveedor

In [6]:
if COLAB_CONTEXT is not None:
    from moderacion_peru.providers import HuggingFaceProvider
    provider=HuggingFaceProvider(model='Qwen/Qwen3-4B',revision='1cfa9a7208912126459214e8b04321603b3df60c',device='cuda',max_new_tokens=512)
    SOURCE=COLAB_CONTEXT.input('chunks_v2')
    OUTPUT=COLAB_CONTEXT.scratch_output_dir/'huggingface_qwen3_4b_v2.jsonl'
    ERRORS=COLAB_CONTEXT.scratch_output_dir/'huggingface_qwen3_4b_v2.errors.jsonl'
else:
    from moderacion_peru.providers import OllamaProvider
    provider=OllamaProvider(model='qwen3.5:4b')
    SOURCE=ROOT/'datos/processed/chunks_v2.jsonl'
    OUTPUT=ROOT/'datos/etiquetado/local/ollama_qwen35_4b_v2.jsonl'
    ERRORS=ROOT/'datos/etiquetado/local/ollama_qwen35_4b_v2.errors.jsonl'
show_result('Estado del proveedor', provider.probe(), tone='success')
show_summary('Rutas de la campaña', {'entrada': SOURCE, 'salida': OUTPUT, 'errores': ERRORS}, tone='neutral')

provider,huggingface_local
model,Qwen/Qwen3-4B
revision,1cfa9a7208912126459214e8b04321603b3df60c
transformers_installed,Sí
hardware,"Ver detalle{ ""backend"": ""cuda"", ""requested"": ""cuda"", ""device_name"": ""NVIDIA L4"", ""torch_version"": ""2.11.0+cu128"", ""runtime_version"": ""12.8"", ""total_memory_bytes"": 23659151360, ""dtype"": ""bfloat16"", ""fallback_reason"": null }"
model_loaded,No


entrada,/content/moderacion_peru/inputs/datos/processed/chunks_v2.jsonl
salida,/content/moderacion_peru/runs/02_01/02_01_working_v2_1/huggingface_qwen3_4b_v2.jsonl
errores,/content/moderacion_peru/runs/02_01/02_01_working_v2_1/huggingface_qwen3_4b_v2.errors.jsonl


## Etiquetado incremental

In [ ]:
from tqdm.auto import tqdm
import inspect
from moderacion_peru.io import read_jsonl
from moderacion_peru.labeling import annotate_incremental

if 'progress_callback' not in inspect.signature(annotate_incremental).parameters:
    raise RuntimeError('El kernel cargó un bundle anterior sin progress_callback. Ejecute 02_00 en Colab, reinicie completamente el kernel y vuelva a empezar por la celda de bootstrap.')

RUN=True  # Cambie a True después de verificar proveedor, entrada y rutas.
LIMIT=None # 20   # Smoke test: procesa como máximo 20 chunks pendientes. None procesa todos los pendientes
# Para procesar TODOS los chunks pendientes use LIMIT=None; no deje el valor en blanco.
# Si mantiene LIMIT=20 y vuelve a ejecutar, continuará con los siguientes 20 porque reanuda por chunk_id.

label_progress={'bar':None}
def report_label_progress(event):
    if event['status']=='started':
        label_progress['bar']=tqdm(total=event['selected'],desc='Etiquetado incremental',unit='chunk')
        return
    bar=label_progress.get('bar')
    if bar is not None and event.get('advance'):
        bar.update(event['advance'])
        bar.set_postfix(etiquetados=event['labeled'],errores=event['errors'])

if RUN:
    try:
        labeling_result=annotate_incremental(read_jsonl(SOURCE),provider,OUTPUT,error_path=ERRORS,limit=LIMIT,progress_callback=report_label_progress)
    finally:
        if label_progress.get('bar') is not None:
            label_progress['bar'].close()
    show_result('Resultado del etiquetado incremental',labeling_result,tone='success')
else:
    show_callout('Preflight completo','Cambie RUN=True. LIMIT=20 ejecuta el piloto; LIMIT=None procesa todo lo pendiente y conserva la reanudación por chunk_id.',tone='neutral')

Etiquetado incremental:   0%|          | 0/166940 [00:00<?, ?chunk/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


## Publicación o checkpoint en Drive

Los archivos se generan en el SSD efímero de `/content`. Active esta celda después de un checkpoint coherente o al finalizar; publica un solo TAR.GZ y luego su manifiesto.

In [ ]:
PUBLISH_TO_DRIVE = True
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación desactivada', 'Cambie PUBLISH_TO_DRIVE=True tras guardar un checkpoint consistente.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] Ollama, "Structured Outputs," Ollama Documentation, 2026. [Online]. Available: https://docs.ollama.com/capabilities/structured-outputs. Accessed: Aug. 5, 2026.

[2] Ollama, "Model Card: qwen3.5:4b," Ollama Model Library, 2026, model digest 2a654d98e6fb. [Online]. Available: https://ollama.com/library/qwen3.5:4b. Accessed: Aug. 5, 2026.

[3] A. Yang, A. Li, B. Yang, et al., "Qwen3 Technical Report," arXiv:2505.09388, 2025, doi: 10.48550/arXiv.2505.09388.

[4] Qwen Team, "Model Card: Qwen/Qwen3-4B," Hugging Face Hub, revision 1cfa9a7208912126459214e8b04321603b3df60c, 2025. [Online]. Available: https://huggingface.co/Qwen/Qwen3-4B/tree/1cfa9a7208912126459214e8b04321603b3df60c. Accessed: Aug. 5, 2026.

[5] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[6] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[7] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.